In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation
from visualization.src.utils import BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS, ARCHITECUTURE_FAMILY_COLORS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "pretraining_results_with_metadata.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all

In [ ]:
METRIC = 'pearsonr_nc'

In [ ]:
architectures = [
    'AlexNet',
    'CORnet-S',
    'EfficientNet',
    'ResNet',
    'ConvNeXt',
    'ViT',
    # 'architecture_average',
]

benchmarks = [
    'bs_fz',
    'bs_mh',
    'tvsd',
    'things_fmri',
    'nsd',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
    # 'benchmark_average',
]



In [ ]:
subfigsize = (7.5, 7)
subfigsize = (9, 7)
subfigsize = (9, 6)
subfigsize = (8, 5)

fig_multiplier = 2.0
fig_multiplier = 1.0
fig_multiplier = 0.75
figsize = (10, 8)

In [ ]:
subfigsize = (7.5, 7)
subfigsize = (9, 7)
subfigsize = (9, 6)
subfigsize = (8, 5)

fig_multiplier = 2.0
fig_multiplier = 1.0
fig_multiplier = 0.75
figsize = (10, 8)
# figsize = (10, 7)

nrows, ncols = 2, 3


figsize = (subfigsize[0] * fig_multiplier * ncols, subfigsize[1] * fig_multiplier * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

for i, arch in enumerate(architectures):

    ax = axes[i // ncols, i % ncols]

    data = df_all.copy()
    data = data[(data.is_adversarially_finetuned != True) & (data.is_selfsupervised_learning != True)]
    data = data[data.model_id.apply(lambda x: 'flex' not in x.lower())]
    data = data[data.backbone_arch_family == arch]
    # print(data)
    # data = data[data.backbone_arch_family == 'ViT']

    data = apply_hiearchical_aggregation(
        df=data,
        groupby_cols=GENERIC_GROUPING_COLUMNS + ['benchmark_name'],
        agg_cols=METRICS_COMPACT,
        agg_func='mean'
    )


    data["Architecture Family"] = data['backbone_arch_family']
    data['Model Source'] = data['backbone_source']
    data["Benchmark"] = data['benchmark_name'].map(BENCHMARK_NAME_MAPPING)

    arch_families = data['Architecture Family'].unique()
    n_arch = len(arch_families)


    # ---- spvvs: lines + markers ----
    sns.lineplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='pretraining_n_samples',
        y=METRIC,
        hue='Benchmark',
        # marker='o',
        marker=None,
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,   # we will build legends manually
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='pretraining_n_samples',
        y=METRIC,
        hue='Benchmark',
        marker='o',
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        alpha=0.3,
        legend=False,   # we will build legends manually
    )

    # ---- timm: markers only ----
    sns.lineplot(
        data=data[data['Model Source'] == 'timm'],
        x='pretraining_n_samples',
        y=METRIC,
        hue='Benchmark',
        marker=None,
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'timm'],
        x='pretraining_n_samples',
        y=METRIC,
        hue='Benchmark',
        marker='X',
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        # errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )


    
    color_handles = [
        Line2D(
            [0], [0],
            color=BENCHMARK_COLORS[benchmark],
            lw=3,
            label=BENCHMARK_NAME_MAPPING[benchmark],
        )
        for i, benchmark in enumerate(benchmarks)
    ]

    color_legend = ax.legend(
        handles=color_handles,
        title="Benchmarks",
        frameon=True,
        loc="center right",
        bbox_to_anchor=(1.3, 0.8)
    )


    source_handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='-',
            color='black',
            markersize=8,
            label='spvvs',
        ),
        Line2D(
            [0], [0],
            marker='X',
            linestyle='--',
            color='black',
            markersize=8,
            label='timm',
        ),
    ]
    if i == len(architectures) - 1:
        ax.legend(
            handles=source_handles,
            title="Model source",
            frameon=True,
            loc="lower right",
            bbox_to_anchor=(1.3, 0.0)
        )
        ax.add_artist(color_legend)
    else:
        ax.legend().remove()


    # ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
    ax.set_xlabel('Pretraining Samples (D)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines[['right', 'top']].set_visible(False)
        
    ax.set_title(f"{arch}", fontsize=16, fontweight='bold')
    
    # ax.legend().remove()
    
plt.suptitle("Alignment Score vs. Pretraining Samples across Architecture Families", fontsize=18, fontweight='bold')
    
plt.tight_layout()

figures_dir = save_dir
fig_name = 'pretraining-samples-no_fit'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)


In [ ]:

# figsize = (10, 7)

nrows, ncols = 2, 3

arch_map = {
    'ResNetFlex': 'ResNet',
    'ConvNeXtFlex': 'ConvNeXt',
    'ViTFlex': 'ViT',
}


figsize = (subfigsize[0] * fig_multiplier * ncols, subfigsize[1] * fig_multiplier * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

for i, arch in enumerate(architectures):

    ax = axes[i // ncols, i % ncols]

    data = df_all.copy()
    data = data[(data.is_adversarially_finetuned != True) & (data.is_selfsupervised_learning != True)]
    # data = data[data.model_id.apply(lambda x: 'flex' not in x.lower())]
    data.backbone_arch_family = data.backbone_arch_family.apply(
        lambda x: arch_map.get(x, x)
    )
    data = data[
        (data.pretraining_samples_per_class == 0)
        | (data.pretraining_samples_per_class.isna())
    ]
    data = data[data.backbone_arch_family == arch]
    # print(data)
    # data = data[data.backbone_arch_family == 'ViT']

    data = apply_hiearchical_aggregation(
        df=data,
        groupby_cols=GENERIC_GROUPING_COLUMNS + ['benchmark_name'],
        agg_cols=METRICS_COMPACT,
        agg_func='mean'
    )


    data["Architecture Family"] = data['backbone_arch_family']
    data['Model Source'] = data['backbone_source']
    data["Benchmark"] = data['benchmark_name'].map(BENCHMARK_NAME_MAPPING)

    arch_families = data['Architecture Family'].unique()
    n_arch = len(arch_families)


    # ---- spvvs: lines + markers ----
    sns.lineplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='backbone_n_params',
        y=METRIC,
        hue='Benchmark',
        # marker='o',
        marker=None,
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,   # we will build legends manually
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='backbone_n_params',
        y=METRIC,
        hue='Benchmark',
        marker='o',
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        alpha=0.3,
        legend=False,   # we will build legends manually
    )

    # ---- timm: markers only ----
    sns.lineplot(
        data=data[data['Model Source'] == 'timm'],
        x='backbone_n_params',
        y=METRIC,
        hue='Benchmark',
        marker=None,
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'timm'],
        x='backbone_n_params',
        y=METRIC,
        hue='Benchmark',
        marker='X',
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        # errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )


    
    color_handles = [
        Line2D(
            [0], [0],
            color=BENCHMARK_COLORS[benchmark],
            lw=3,
            label=BENCHMARK_NAME_MAPPING[benchmark],
        )
        for i, benchmark in enumerate(benchmarks)
    ]

    color_legend = ax.legend(
        handles=color_handles,
        title="Benchmarks",
        frameon=True,
        loc="center right",
        bbox_to_anchor=(1.3, 0.8)
    )


    source_handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='-',
            color='black',
            markersize=8,
            label='spvvs',
        ),
        Line2D(
            [0], [0],
            marker='X',
            linestyle='--',
            color='black',
            markersize=8,
            label='timm',
        ),
    ]
    if i == len(architectures) - 1:
        ax.legend(
            handles=source_handles,
            title="Model source",
            frameon=True,
            loc="lower right",
            bbox_to_anchor=(1.3, 0.0)
        )
        ax.add_artist(color_legend)
    else:
        ax.legend().remove()


    # ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
    ax.set_xlabel('Model Parameters (N)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines[['right', 'top']].set_visible(False)
        
    ax.set_title(f"{arch}", fontsize=16, fontweight='bold')
    
    # ax.legend().remove()
    
plt.suptitle("Alignment Score vs. Backbone Parameters across Architecture Families", fontsize=18, fontweight='bold')
    
plt.tight_layout()

figures_dir = save_dir
fig_name = 'pretraining-parameters-no_fit'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)


In [ ]:

# figsize = (10, 7)

nrows, ncols = 2, 3

arch_map = {
    'ResNetFlex': 'ResNet',
    'ConvNeXtFlex': 'ConvNeXt',
    'ViTFlex': 'ViT',
}


figsize = (subfigsize[0] * fig_multiplier * ncols, subfigsize[1] * fig_multiplier * nrows)

fig, axes = plt.subplots(
    nrows=nrows, 
    ncols=ncols, 
    figsize=figsize, 
    # sharey=True,
    dpi = 300
    )

for i, arch in enumerate(architectures):

    ax = axes[i // ncols, i % ncols]

    data = df_all.copy()
    data = data[(data.is_adversarially_finetuned != True) & (data.is_selfsupervised_learning != True)]
    # data = data[data.model_id.apply(lambda x: 'flex' not in x.lower())]
    data.backbone_arch_family = data.backbone_arch_family.apply(
        lambda x: arch_map.get(x, x)
    )
    # data = data[
    #     (data.pretraining_samples_per_class == 0)
    #     | (data.pretraining_samples_per_class.isna())
    # ]
    data = data[data.backbone_arch_family == arch]
    # print(data)
    # data = data[data.backbone_arch_family == 'ViT']

    data = apply_hiearchical_aggregation(
        df=data,
        groupby_cols=GENERIC_GROUPING_COLUMNS + ['benchmark_name'],
        agg_cols=METRICS_COMPACT,
        agg_func='mean'
    )


    data["Architecture Family"] = data['backbone_arch_family']
    data['Model Source'] = data['backbone_source']
    data["Benchmark"] = data['benchmark_name'].map(BENCHMARK_NAME_MAPPING)

    arch_families = data['Architecture Family'].unique()
    n_arch = len(arch_families)


    # ---- spvvs: lines + markers ----
    sns.lineplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='pretraining_total_flops_estimate',
        y=METRIC,
        hue='Benchmark',
        # marker='o',
        marker=None,
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,   # we will build legends manually
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'spvvs'],
        x='pretraining_total_flops_estimate',
        y=METRIC,
        hue='Benchmark',
        marker='o',
        linestyle='-',
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        alpha=0.3,
        legend=False,   # we will build legends manually
    )

    # ---- timm: markers only ----
    sns.lineplot(
        data=data[data['Model Source'] == 'timm'],
        x='pretraining_total_flops_estimate',
        y=METRIC,
        hue='Benchmark',
        marker=None,
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )
    sns.scatterplot(
        data=data[data['Model Source'] == 'timm'],
        x='pretraining_total_flops_estimate',
        y=METRIC,
        hue='Benchmark',
        marker='X',
        # linestyle='None',
        linestyle='--',
        # markersize=10,
        # errorbar=None,
        ax=ax,
        palette={v: BENCHMARK_COLORS[k] for k, v in BENCHMARK_NAME_MAPPING.items()},
        legend=False,
    )


    
    color_handles = [
        Line2D(
            [0], [0],
            color=BENCHMARK_COLORS[benchmark],
            lw=3,
            label=BENCHMARK_NAME_MAPPING[benchmark],
        )
        for i, benchmark in enumerate(benchmarks)
    ]

    color_legend = ax.legend(
        handles=color_handles,
        title="Benchmarks",
        frameon=True,
        loc="center right",
        bbox_to_anchor=(1.3, 0.8)
    )


    source_handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='-',
            color='black',
            markersize=8,
            label='spvvs',
        ),
        Line2D(
            [0], [0],
            marker='X',
            linestyle='--',
            color='black',
            markersize=8,
            label='timm',
        ),
    ]
    if i == len(architectures) - 1:
        ax.legend(
            handles=source_handles,
            title="Model source",
            frameon=True,
            loc="lower right",
            bbox_to_anchor=(1.3, 0.0)
        )
        ax.add_artist(color_legend)
    else:
        ax.legend().remove()


    # ax.set_title(benchmark_v, fontsize=16, fontweight='bold')
    ax.set_xlabel('Pretraining Samples (D)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Alignment Score', fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.2, 0.4, 0.6, 0.8, 1.0])
    ax.spines[['right', 'top']].set_visible(False)
        
    ax.set_title(f"{arch}", fontsize=16, fontweight='bold')
    
    # ax.legend().remove()
    
plt.suptitle("Alignment Score vs. Backbone Parameters across Architecture Families", fontsize=18, fontweight='bold')
    
plt.tight_layout()

# figures_dir = save_dir
# fig_name = 'pretraining-compute-no_fit'
# formats = ['pdf', 'png', 'svg']
# save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)
